In [ ]:
# import pdal
import json
import os
import rasterio
import numpy as np
import laspy
import pandas as pd

In [ ]:
outfile_subset_dir = "/home/ajai-krishna/work/GEO_AI/outputs/subsets"

In [ ]:
las_file = laspy.open('/home/ajai-krishna/work/GEO_AI/data/input/laz/DEVDI_POINT CLOUD (511671).las')
header = las_file.header

In [ ]:
variables = list(las_file.header.point_format.dimension_names) 
variables

In [ ]:
window_size = 10
chunk_size = 1000000

In [ ]:
x_min, y_min, z_min = header.min
x_max, y_max, z_max = header.max
x_windows = np.arange(x_min, x_max, window_size)
y_windows = np.arange(y_min, y_max, window_size)

In [ ]:
# split LAS into spatial windows and write subsets
from pathlib import Path

output_dir = outfile_subset_dir  # defined earlier in the notebook

# iterate over raster windows computed above
for wx in x_windows:
    for wy in y_windows:
        xmin, xmax = wx, wx + window_size
        ymin, ymax = wy, wy + window_size

        selected_points = []

        # read the file in chunks and collect points inside the current window
        for chunk in las_file.chunk_iterator(chunk_size):
            x = chunk.x
            y = chunk.y
            mask = (
                (x >= xmin) & (x < xmax) &
                (y >= ymin) & (y < ymax)
            )
            if np.any(mask):
                selected_points.append(chunk[mask].copy())

        if len(selected_points) == 0:
            # no points in this window, skip
            continue

        # Merge selected chunks into a single array
        points = np.concatenate(selected_points)

        # Create a fresh header for the output
        out_header = laspy.LasHeader(
            point_format=header.point_format,
            version=header.version
        )
        out_header.scales = header.scales
        out_header.offsets = header.offsets

        out_las = laspy.LasData(out_header)
        out_las.points = points

        out_name = f"{Path(las_file).stem}_{int(wx)}_{int(wy)}.las"
        out_path = os.path.join(output_dir, out_name)
        out_las.write(out_path)
        print("Saved:", out_path)


In [ ]:
# previous attempt at chunk/window iteration was misaligned and is now replaced by the cell above.
# see the cell immediately before this one for the corrected implementation.


In [ ]:
df1